# VideoToSMPL — Notebook 03: End-to-end pipeline

**Video in → G1 PKL + preview MP4 out.**

Colab free T4 · zero install · single person, static camera, ≤ 30 s clip recommended.

1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → **Run all**
3. Upload video when prompted

## 1a. Clone + torch pin

In [ ]:
%cd /content
import sys, os
print(f'Python {sys.version.split()[0]}')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

![ -d GVHMR ] || git clone --depth 1 https://github.com/zju3dv/GVHMR.git
![ -d GMR ]   || git clone --depth 1 https://github.com/YanjieZe/GMR.git

# GVHMR's pin — torch 2.3.0 + CUDA 12.1
!pip install -q torch==2.3.0+cu121 torchvision==0.18.0+cu121 --index-url https://download.pytorch.org/whl/cu121

import torch
assert torch.cuda.is_available(), 'CUDA not visible — switch runtime to T4 GPU'
print(f'✓ torch {torch.__version__} · CUDA {torch.version.cuda} · {torch.cuda.get_device_name(0)}')

## 1b. pytorch3d

GVHMR pins pytorch3d 0.7.6. A prebuilt wheel only exists for **Python 3.10 + CUDA 12.1 + torch 2.3.0**. Colab's default Python changes over time — if it isn't 3.10 we build from source (~8–12 min, verbose so failures are actionable).

In [ ]:
import sys, os, subprocess
py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'

if py_tag == 'cp310':
    print('→ installing prebuilt pytorch3d wheel (py310)')
    !pip install https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py310_cu121_pyt230/pytorch3d-0.7.6-cp310-cp310-linux_x86_64.whl
else:
    print(f'→ Colab is {py_tag}; building pytorch3d 0.7.6 from source')
    # Build prerequisites
    !pip install -q ninja fvcore iopath
    # Colab's CUDA toolkit lives here — matches the torch 2.3.0+cu121 build
    os.environ['CUDA_HOME'] = os.environ.get('CUDA_HOME', '/usr/local/cuda')
    os.environ['FORCE_CUDA'] = '1'
    os.environ['MAX_JOBS'] = '4'
    print(f'  CUDA_HOME = {os.environ["CUDA_HOME"]}')
    if os.path.exists(os.environ['CUDA_HOME']):
        !ls -d $CUDA_HOME 2>/dev/null && $CUDA_HOME/bin/nvcc --version 2>/dev/null | tail -1
    else:
        print('  ⚠ CUDA_HOME missing — build will likely fail')
    # NOT quiet — on failure the real error is the last ~30 lines
    !pip install --no-build-isolation "git+https://github.com/facebookresearch/pytorch3d.git@V0.7.6"
    try:
        import pytorch3d  # noqa: F401
        print('✓ pytorch3d built + imports cleanly')
    except ImportError as e:
        print(f'\n✗ pytorch3d build FAILED: {e}')
        print('Diagnostics:')
        !echo '  nvcc:' && $CUDA_HOME/bin/nvcc --version 2>&1 | tail -1 || echo '    (not found)'
        !echo '  torch CUDA:' && python -c 'import torch; print(f"    {torch.version.cuda}, arch={torch.cuda.get_arch_list()}")'
        raise

## 1c. Remaining GVHMR deps + GMR + rendering

In [ ]:
%cd /content
# Strip pytorch3d URL from requirements (we installed it separately)
!grep -v 'pytorch3d' GVHMR/requirements.txt > /tmp/gvhmr_reqs.txt
!pip install -q -r /tmp/gvhmr_reqs.txt
# demo.py imports `pytorch_lightning`; `lightning` alone doesn't always alias it
!pip install -q pytorch-lightning==2.3.0
!pip install -q -e GVHMR --no-deps
!pip install -q -e GMR
!pip install -q 'mujoco>=3.1' 'imageio[ffmpeg]' scipy smplx

# Final smoke test
import pytorch_lightning, pytorch3d  # noqa: F401
print('✓ pytorch_lightning + pytorch3d importable — setup complete')

## 2. Download GVHMR weights

In [ ]:
import urllib.request
from pathlib import Path
CKPT = Path('/content/GVHMR/inputs/checkpoints')
for sub in ['gvhmr','hmr2','vitpose','yolo','body_models/smpl','body_models/smplx']:
    (CKPT/sub).mkdir(parents=True, exist_ok=True)
WEIGHTS = {
  'gvhmr/gvhmr_siga24_release.ckpt':'https://huggingface.co/camenduru/GVHMR/resolve/main/gvhmr/gvhmr_siga24_release.ckpt',
  'hmr2/epoch=10-step=25000.ckpt':'https://huggingface.co/camenduru/GVHMR/resolve/main/hmr2/epoch%3D10-step%3D25000.ckpt',
  'vitpose/vitpose-h-multi-coco.pth':'https://huggingface.co/camenduru/GVHMR/resolve/main/vitpose/vitpose-h-multi-coco.pth',
  'yolo/yolov8x.pt':'https://huggingface.co/camenduru/GVHMR/resolve/main/yolo/yolov8x.pt',
  'body_models/smpl/SMPL_NEUTRAL.pkl':'https://huggingface.co/camenduru/SMPLer-X/resolve/main/SMPL_NEUTRAL.pkl',
}
for rel, url in WEIGHTS.items():
    dest = CKPT / rel
    if dest.exists() and dest.stat().st_size > 1024:
        print(f'✓ cached {dest.name}')
        continue
    print(f'↓ {dest.name}')
    urllib.request.urlretrieve(url, dest)
print('\n✓ Public weights ready.')

## 2b. SMPL-X (optional but required for retargeting)

License-gated — can't auto-download. Get it:
1. Register at https://smpl-x.is.tue.mpg.de/
2. Download **SMPL-X v1.1 (NPZ)** → extract `SMPLX_NEUTRAL.npz`

Upload it now — or press Cancel to run **extraction only** (cell 5 retarget will auto-skip).

In [ ]:
from pathlib import Path
import shutil

SMPLX_TARGETS = [
    Path('/content/GVHMR/inputs/checkpoints/body_models/smplx/SMPLX_NEUTRAL.npz'),
    Path('/content/GMR/assets/body_models/smplx/SMPLX_NEUTRAL.npz'),
]
HAVE_SMPLX = any(p.exists() and p.stat().st_size > 1024 for p in SMPLX_TARGETS)

if HAVE_SMPLX:
    src = next(p for p in SMPLX_TARGETS if p.exists())
    for p in SMPLX_TARGETS:
        if not p.exists():
            p.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, p)
    print('✓ SMPL-X already cached — retargeting enabled')
else:
    print('Upload SMPLX_NEUTRAL.npz  (press Cancel to skip — extraction still works)')
    from google.colab import files
    try:
        up = files.upload()
    except Exception as e:
        print(f'upload cancelled: {e}')
        up = {}
    if not up:
        print('⚠ SMPL-X not uploaded — cell 5 (retarget) will be skipped.')
    else:
        fname = list(up.keys())[0]
        blob = up[fname]
        for p in SMPLX_TARGETS:
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_bytes(blob)
        HAVE_SMPLX = True
        print(f'✓ saved SMPLX_NEUTRAL.npz ({len(blob)/1e6:.1f} MB) — retargeting enabled')

## 3. Upload your video

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()
assert uploaded, 'No video uploaded — run this cell again and pick a file.'
video = Path('/content') / list(uploaded.keys())[0]
print(f'{video.name}  ({video.stat().st_size/1e6:.1f} MB)')

## 4. Extract SMPL (GVHMR)

In [ ]:
import subprocess, time
from pathlib import Path

%cd /content/GVHMR
t0 = time.time()
r = subprocess.run(['python', 'tools/demo/demo.py', '--video', str(video), '-s'],
                   capture_output=True, text=True)
print('\n'.join(r.stdout.splitlines()[-15:]))
if r.returncode != 0:
    print('STDERR:\n' + r.stderr[-3000:])
    raise RuntimeError(f'GVHMR failed ({r.returncode})')
pt = Path('/content/GVHMR/outputs/demo') / video.stem / 'hmr4d_results.pt'
assert pt.exists(), f'missing {pt}'
print(f'\n✓ GVHMR {time.time()-t0:.0f}s → {pt.name}  ({pt.stat().st_size/1e6:.1f} MB)')

## 5. Retarget to Unitree G1 (GMR) — requires SMPL-X

In [ ]:
import subprocess, time
from pathlib import Path

if not HAVE_SMPLX:
    print('⏭  Skipping retarget — SMPL-X not uploaded in cell 2b.')
    pkl = None
else:
    %cd /content/GMR
    pkl = Path('/content') / (video.stem + '_g1.pkl')
    t0 = time.time()
    r = subprocess.run(['python', 'scripts/gvhmr_to_robot.py',
                        '--gvhmr_pt', str(pt),
                        '--robot', 'unitree_g1',
                        '--save_path', str(pkl)],
                       capture_output=True, text=True)
    print('\n'.join(r.stdout.splitlines()[-15:]))
    if r.returncode != 0:
        print('STDERR:\n' + r.stderr[-3000:])
        raise RuntimeError(f'GMR failed ({r.returncode})')
    print(f'\n✓ Retarget {time.time()-t0:.0f}s → {pkl.name}')

## 6. Render MuJoCo preview

In [ ]:
import os; os.environ.setdefault('MUJOCO_GL', 'egl')
import pickle, numpy as np
from pathlib import Path

preview = None
if pkl is None:
    print('⏭  No PKL — skipping render.')
else:
    import imageio.v2 as imageio, mujoco
    from general_motion_retargeting import ROBOT_XML_DICT
    with open(pkl, 'rb') as f: d = pickle.load(f)
    model = mujoco.MjModel.from_xml_path(str(ROBOT_XML_DICT['unitree_g1']))
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model, height=720, width=1280)
    cam = mujoco.MjvCamera(); cam.distance=3.0; cam.azimuth=90; cam.elevation=-15; cam.lookat[:]=[0,0,0.9]
    preview = Path('/content') / (video.stem + '_g1.mp4')
    w = imageio.get_writer(str(preview), fps=int(d['fps']), codec='libx264', quality=8)
    for i in range(len(d['root_pos'])):
        qw,qx,qy,qz = d['root_rot'][i][[3,0,1,2]]
        data.qpos[:3] = d['root_pos'][i]
        data.qpos[3:7] = [qw, qx, qy, qz]
        data.qpos[7:] = d['dof_pos'][i]
        mujoco.mj_forward(model, data)
        renderer.update_scene(data, camera=cam)
        w.append_data(np.asarray(renderer.render()))
    w.close(); renderer.close()
    print(f'✓ preview → {preview.name}  ({preview.stat().st_size/1e6:.1f} MB)')

## 7. Download artifacts

In [ ]:
from google.colab import files
files.download(str(pt))
if pkl:     files.download(str(pkl))
if preview: files.download(str(preview))